## Optimization and evaluation design

The implementation uses a pretrained encoder with a task-specific classification head. The frozen baseline trains only the head; full fine-tuning releases the encoder and uses smaller updates for pretrained parameters than for the new head. Fixed seeds, AdamW weight decay, linear warmup/decay, gradient clipping, validation-based model selection, a shared held-out test split, per-class reporting, and real-world/domain-shift checks are used to support a reproducible and interpretable experiment. The model-specific training procedure is implemented for BERT rather than copying an image-classification architecture.


In [ ]:
!pip -q install "datasets>=2.18.0" "transformers>=4.40.0" "accelerate>=0.28.0" "scikit-learn>=1.4.0" "shap>=0.45.0" pandas matplotlib seaborn

In [ ]:
import os, gc, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Stream the complete Civil Comments dataset and cache locally

This notebook uses all valid rows from the official `google/civil_comments` train, validation, and test splits. It does not sample, truncate, or manually upload a dataset.

On the first run, Hugging Face Datasets streams each split and writes it in batches to local Parquet files:
- `civil_comments_full_cache/train.parquet`
- `civil_comments_full_cache/validation.parquet`
- `civil_comments_full_cache/test.parquet`

Later runs reuse the Parquet cache and skip remote collection. Keep the cache folder on persistent storage. In Colab, mount Google Drive and set `CACHE_DIR` to a folder in Drive if you need the cache to survive runtime resets.

The official train/validation/test split structure is preserved. Streaming bounds data-loading memory, but full-dataset BERT training is still compute-intensive.


In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset, Dataset

DATASET_NAME = "google/civil_comments"
MODEL_NAME = "bert-base-uncased"
TOXICITY_THRESHOLD = 0.5
MAX_LENGTH = 128
SEED = 42
PARQUET_BATCH_SIZE = 5_000

# For persistent Colab cache, mount Drive first and change this path, e.g.
# CACHE_DIR = Path("/content/drive/MyDrive/civil_comments_full_cache")
CACHE_DIR = Path("civil_comments_full_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILES = {
    "train": CACHE_DIR / "train.parquet",
    "validation": CACHE_DIR / "validation.parquet",
    "test": CACHE_DIR / "test.parquet",
}

def stream_split_to_parquet(split_name, output_path, batch_size=PARQUET_BATCH_SIZE):
    """Stream every valid row in a split to Parquet without loading the split into RAM."""
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Using existing cache: {output_path}")
        return

    temp_path = output_path.with_suffix(".parquet.incomplete")
    if temp_path.exists():
        temp_path.unlink()

    stream = load_dataset(DATASET_NAME, split=split_name, streaming=True)
    writer, pending, total = None, [], 0
    try:
        for row in stream:
            text_value = row.get("text")
            toxicity_value = row.get("toxicity")
            if text_value is None or not str(text_value).strip() or toxicity_value is None:
                continue
            score = float(toxicity_value)
            pending.append({
                "text": str(text_value),
                "toxicity": score,
                "label": int(score >= TOXICITY_THRESHOLD),
            })
            if len(pending) >= batch_size:
                table = pa.Table.from_pylist(pending)
                if writer is None:
                    writer = pq.ParquetWriter(str(temp_path), table.schema, compression="snappy")
                writer.write_table(table)
                total += len(pending)
                pending.clear()
                print(f"{split_name}: cached {total:,} rows")
        if pending:
            table = pa.Table.from_pylist(pending)
            if writer is None:
                writer = pq.ParquetWriter(str(temp_path), table.schema, compression="snappy")
            writer.write_table(table)
            total += len(pending)
    finally:
        if writer is not None:
            writer.close()

    if total == 0 or not temp_path.exists():
        raise RuntimeError(f"No valid rows were written for split: {split_name}")
    temp_path.replace(output_path)
    print(f"Finished {split_name}: {total:,} rows -> {output_path}")

# First run streams every official split. Later runs reuse the local Parquet files.
for split_name, output_path in SPLIT_FILES.items():
    stream_split_to_parquet(split_name, output_path)

# Hugging Face's Parquet loader uses Arrow-backed data rather than a pandas copy.
train_ds = load_dataset("parquet", data_files=str(SPLIT_FILES["train"]), split="train")
valid_ds = load_dataset("parquet", data_files=str(SPLIT_FILES["validation"]), split="train")
test_ds = load_dataset("parquet", data_files=str(SPLIT_FILES["test"]), split="train")

for name, ds in [("train", train_ds), ("validation", valid_ds), ("test", test_ds)]:
    labels = pd.Series(ds["label"])
    print(name, f"{len(ds):,} rows",
          labels.value_counts(normalize=True).sort_index().to_dict())


## 2. Tokenization and PyTorch datasets


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        row = self.dataset[idx]
        enc = tokenizer(
            row["text"], truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

def make_loader(ds, batch_size, shuffle=False):
    return DataLoader(CommentDataset(ds), batch_size=batch_size, shuffle=shuffle,
                      num_workers=2 if DEVICE.type == "cuda" else 0,
                      pin_memory=(DEVICE.type == "cuda"))

## 3. Model and evaluation helpers

The classifier uses the final hidden state of the `[CLS]` token followed by dropout and a two-class linear layer.

In [ ]:
class BertToxicityClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

def compute_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": p, "recall": r, "f1": f1}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_y, all_pred, all_prob = [], [], []
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        logits = model(ids, mask)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs >= 0.5).long()
        all_y.extend(batch["labels"].numpy().tolist())
        all_pred.extend(preds.cpu().numpy().tolist())
        all_prob.extend(probs.cpu().numpy().tolist())
    return compute_metrics(all_y, all_pred), np.array(all_y), np.array(all_pred), np.array(all_prob)

def train_model(model, train_loader, val_loader, epochs=2, lr=2e-5, frozen=False):
    """Train head-only baseline or full BERT with differential learning rates."""
    model.to(DEVICE)
    if frozen:
        for p in model.bert.parameters():
            p.requires_grad = False
        optimizer = torch.optim.AdamW(
            model.classifier.parameters(), lr=lr, weight_decay=0.01
        )
    else:
        for p in model.bert.parameters():
            p.requires_grad = True
        # Differential learning-rate strategy adapted to BERT:
        # gentle updates to pretrained layers, faster learning for the new head.
        last_layer = model.bert.encoder.layer[-1]
        last_layer_ids = {id(p) for p in last_layer.parameters()}
        early_params = [p for p in model.bert.parameters() if id(p) not in last_layer_ids]
        optimizer = torch.optim.AdamW([
            {"params": early_params, "lr": lr},
            {"params": last_layer.parameters(), "lr": lr * 2.0},
            {"params": model.classifier.parameters(), "lr": lr * 5.0},
        ], weight_decay=0.01)

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(0.1 * total_steps)),
        num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()
    history = []
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(ids, mask)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item()
        val_metrics, *_ = evaluate(model, val_loader)
        history.append({"epoch": epoch+1, "train_loss": running/len(train_loader), **val_metrics})
        print(history[-1])
    return history


## 4. Transfer learning baseline: frozen BERT feature extractor

Only the classification head is trained. BERT weights remain frozen. This is the required pre-tuning baseline.

In [ ]:
BASELINE_BATCH_SIZE = 16
baseline_train_loader = make_loader(train_ds, BASELINE_BATCH_SIZE, shuffle=True)
baseline_val_loader = make_loader(valid_ds, BASELINE_BATCH_SIZE)

baseline_model = BertToxicityClassifier()
baseline_history = train_model(
    baseline_model, baseline_train_loader, baseline_val_loader,
    epochs=2, lr=1e-3, frozen=True
)
baseline_metrics, y_val, pred_val, prob_val = evaluate(baseline_model, baseline_val_loader)
print("Frozen baseline validation:", baseline_metrics)

## 5. End-to-end fine-tuning

All BERT parameters are unfrozen and updated together with the classification head.

In [ ]:
FT_BATCH_SIZE = 16
ft_train_loader = make_loader(train_ds, FT_BATCH_SIZE, shuffle=True)
ft_val_loader = make_loader(valid_ds, FT_BATCH_SIZE)

finetuned_model = BertToxicityClassifier()
finetune_history = train_model(
    finetuned_model, ft_train_loader, ft_val_loader,
    epochs=2, lr=2e-5, frozen=False
)
finetuned_metrics, *_ = evaluate(finetuned_model, ft_val_loader)
print("Fine-tuned validation:", finetuned_metrics)

## 6. Hyperparameter tuning

A small grid search tests the BERT backbone learning rate and batch size. During each full fine-tuning trial, the final BERT encoder layer uses 2× the backbone learning rate and the new classification head uses 5×; earlier pretrained layers use the base rate. This assigns gentler updates to pretrained parameters and a larger learning rate to the new classification head. The validation F1 selects the best configuration. The test set is not used for tuning.


In [ ]:
search_space = [
    {"learning_rate": 2e-5, "batch_size": 16},
    {"learning_rate": 3e-5, "batch_size": 16},
    {"learning_rate": 2e-5, "batch_size": 32},
    {"learning_rate": 3e-5, "batch_size": 32},
]
tuning_rows = []

for trial, hp in enumerate(search_space, start=1):
    print(f"Trial {trial}: {hp}")
    trial_model = BertToxicityClassifier()
    tr_loader = make_loader(train_ds, hp["batch_size"], shuffle=True)
    va_loader = make_loader(valid_ds, hp["batch_size"])
    hist = train_model(trial_model, tr_loader, va_loader, epochs=2,
                       lr=hp["learning_rate"], frozen=False)
    metrics, *_ = evaluate(trial_model, va_loader)
    tuning_rows.append({**hp, **metrics})
    del trial_model, tr_loader, va_loader
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

tuning_df = pd.DataFrame(tuning_rows).sort_values("f1", ascending=False)
display(tuning_df)
os.makedirs("results", exist_ok=True)
tuning_df.to_csv("results/hyperparameter_tuning.csv", index=False)

plt.figure(figsize=(8, 4))
labels = [f"lr={r.learning_rate:g}, bs={r.batch_size}" for r in tuning_df.itertuples()]
plt.bar(labels, tuning_df["f1"])
plt.ylabel("Validation F1")
plt.xlabel("Hyperparameter configuration")
plt.title("Hyperparameter tuning results")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("results/hyperparameter_tuning.png", dpi=180)
plt.show()

best = tuning_df.iloc[0].to_dict()
print("Best configuration:", best)

In [ ]:
# Select hyperparameters using validation F1, then retrain a fresh model.
best = tuning_df.iloc[0].to_dict()
best_lr = float(best["learning_rate"])
best_bs = int(best["batch_size"])
print("Selected learning rate:", best_lr, "| batch size:", best_bs)
final_train_loader = make_loader(train_ds, best_bs, shuffle=True)
final_val_loader = make_loader(valid_ds, best_bs)
final_model = BertToxicityClassifier()
final_history = train_model(final_model, final_train_loader, final_val_loader,
                           epochs=2, lr=best_lr, frozen=False)
final_val_metrics, *_ = evaluate(final_model, final_val_loader)
print("Final tuned model validation:", final_val_metrics)


## 7. Train the final model using the best hyperparameters

The next code cell selects the best validation-F1 configuration and trains a fresh final model. The held-out test split is evaluated only after this selection.


In [ ]:
# Evaluate every model stage on the SAME held-out test split for a fair comparison.
test_loader = make_loader(test_ds, best_bs)
baseline_test_metrics, y_test, baseline_test_pred, baseline_test_prob = evaluate(
    baseline_model, test_loader
)
initial_ft_test_metrics, _, initial_ft_test_pred, initial_ft_test_prob = evaluate(
    finetuned_model, test_loader
)
test_metrics, _, pred_test, prob_test = evaluate(final_model, test_loader)

print("Frozen baseline test:", baseline_test_metrics)
print("Initial fine-tuned test:", initial_ft_test_metrics)
print("Tuned fine-tuned test:", test_metrics)
print("\\nFinal tuned model classification report:")
print(classification_report(y_test, pred_test, target_names=["Not toxic", "Toxic"], zero_division=0))
print("Confusion matrix:\\n", confusion_matrix(y_test, pred_test))

comparison = pd.DataFrame([
    {"model":"Frozen BERT baseline", **baseline_test_metrics},
    {"model":"Fine-tuned BERT (initial)", **initial_ft_test_metrics},
    {"model":"Fine-tuned BERT (tuned)", **test_metrics}
])
display(comparison)
comparison.to_csv("results/model_comparison.csv", index=False)


## 8. Real-world text test

These are illustrative examples written for this assignment, not examples from the test set. Inspect each prediction and discuss whether it is reasonable. A toxicity classifier can make mistakes, especially with sarcasm, reclaimed language, quotations, and context-dependent statements.

In [ ]:
real_world_comments = [
    "Thank you for taking the time to explain your perspective. I respectfully disagree.",
    "You are an idiot and nobody should listen to anything you say.",
    "I think this proposal is poorly designed, and the evidence does not support it."
]

def predict_texts(model, texts):
    model.eval()
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LENGTH,
                    return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(enc["input_ids"], enc["attention_mask"]), dim=1)[:, 1]
    return [{"text": t, "toxicity_probability": float(p),
             "predicted_class": "Toxic" if p >= 0.5 else "Not toxic"}
            for t, p in zip(texts, probs.cpu().tolist())]

real_world_results = predict_texts(final_model, real_world_comments)
display(pd.DataFrame(real_world_results))
pd.DataFrame(real_world_results).to_csv("results/real_world_predictions.csv", index=False)

## 9. Interpretability with SHAP

The notebook explains four selected comments and exports token-level SHAP contributions for inspection. SHAP values are local explanations, not causal proof.


In [ ]:
import shap
from pathlib import Path

SHAP_DIR = Path("results/shap")
SHAP_DIR.mkdir(parents=True, exist_ok=True)

def shap_predict(text_list):
    final_model.eval()
    enc = tokenizer(list(text_list), truncation=True, padding=True,
                    max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = final_model(enc["input_ids"], enc["attention_mask"])
        return torch.softmax(logits, dim=1).cpu().numpy()

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(
    shap_predict, masker, output_names=["Not toxic", "Toxic"]
)

explain_texts = [
    "You are an idiot and nobody should listen to anything you say.",
    "Thank you for sharing your view. I respectfully disagree.",
    "This is a terrible idea and the evidence does not support it.",
    "I appreciate your effort, but I think the explanation is incomplete."
]
shap_values = explainer(explain_texts)

# Display local SHAP text plots and export token-level contributions.
for idx, text in enumerate(explain_texts):
    print(f"SHAP example {idx + 1}: {text}")
    shap.plots.text(shap_values[idx, :, 1])

    tokens = list(shap_values.data[idx])
    values = np.asarray(shap_values.values[idx, :, 1]).reshape(-1)
    n = min(len(tokens), len(values))
    token_df = pd.DataFrame({
        "token": [str(t) for t in tokens[:n]],
        "shap_value_toxic": values[:n]
    })
    token_df.to_csv(SHAP_DIR / f"example_{idx + 1}_token_attributions.csv", index=False)

shap_summary_rows = []
real_preds = predict_texts(final_model, explain_texts)
for idx, result in enumerate(real_preds):
    vals = np.asarray(shap_values.values[idx, :, 1]).reshape(-1)
    toks = [str(t) for t in list(shap_values.data[idx])]
    pairs = sorted(zip(toks[:len(vals)], vals), key=lambda x: abs(float(x[1])), reverse=True)
    shap_summary_rows.append({
        "example": idx + 1,
        "predicted_class": result["predicted_class"],
        "toxicity_probability": result["toxicity_probability"],
        "top_attribution_tokens": "; ".join(
            f"{tok} ({float(val):+.4f})" for tok, val in pairs[:8]
        )
    })
pd.DataFrame(shap_summary_rows).to_csv(
    SHAP_DIR / "explanation_summary.csv", index=False
)


## 10. Save artifacts and final experiment summary

In [ ]:
os.makedirs("results", exist_ok=True)
torch.save(final_model.state_dict(), "results/final_model_state_dict.pt")
summary = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "toxicity_threshold": TOXICITY_THRESHOLD,
    "full_split_sizes": {"train": len(train_ds), "validation": len(valid_ds), "test": len(test_ds)},
    "baseline_validation_metrics": baseline_metrics,
    "initial_finetuning_validation_metrics": finetuned_metrics,
    "best_hyperparameters": {"learning_rate": best_lr, "batch_size": best_bs, "epochs": 2},
    "final_test_metrics": test_metrics
}
with open("results/experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 11. Discussion prompts for the report

1. Did fine-tuning improve validation F1 over the frozen feature-extractor baseline? Explain using your measured results.
2. Which learning rate and batch size achieved the best validation F1? Was the difference large or small?
3. Compare precision and recall. Which error is more problematic for your intended use: false positives or false negatives?
4. What do the SHAP token attributions suggest for the selected examples? Are they linguistically sensible?
5. Discuss limitations: English-only training data, historical comments, subjectivity in toxicity labels, possible demographic bias, and lack of conversational context.
6. Do not present the model as a definitive moderator. Use human review for consequential moderation decisions.